# CARTS Research - Interactive Demo

This notebook demonstrates the key algorithms and evaluation methods from the CARTS research project.

## Overview
- **DART Algorithm**: Difficulty-aware spaced repetition
- **CARTS Framework**: Deep RL for joint optimization
- **Context Transfer Evaluation**: LLM-as-a-Judge methodology
- **Statistical Analysis**: Comprehensive comparison framework


In [ ]:
# Import required libraries
import json
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path

# Set up plotting
plt.style.use('seaborn-v0_8')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12

print("📚 CARTS Research Demo - Libraries Loaded")
print("=" * 50)

## 1. Load Mock Data

We'll use simulated data that matches the structure of the real study data.

In [ ]:
# Load mock experimental data
def load_mock_data():
    """Load simulated data for demonstration"""
    
    # Simulate 50 participants across 6 algorithms
    algorithms = ['SM-2', 'HLR', 'KARL', 'LECTOR', 'DART', 'CARTS']
    n_participants = 50
    n_words = 25
    n_weeks = 2
    
    data = []
    
    for alg_idx, algorithm in enumerate(algorithms):
        for participant in range(n_participants // len(algorithms)):
            for week in range(1, n_weeks + 1):
                # Simulate performance with CARTS being best
                base_performance = 0.6 + (alg_idx * 0.05)  # CARTS gets highest base
                week_improvement = week * 0.1
                noise = np.random.normal(0, 0.1)
                
                accuracy = np.clip(base_performance + week_improvement + noise, 0, 1)
                context_transfer = np.clip(accuracy + np.random.normal(0, 0.05), 0, 1)
                
                data.append({
                    'participant_id': f'{algorithm}_P{participant:02d}',
                    'algorithm': algorithm,
                    'week': week,
                    'accuracy': accuracy,
                    'context_transfer': context_transfer,
                    'response_time': np.random.lognormal(2.5, 0.5),  # ~12 seconds average
                    'proficiency_level': np.random.choice(['A2', 'B1', 'B2'], p=[0.3, 0.5, 0.2])
                })
    
    return pd.DataFrame(data)

# Load the data
df = load_mock_data()
print(f"📊 Loaded mock data: {len(df)} observations")
print(f"👥 Participants: {df['participant_id'].nunique()}")
print(f"🔬 Algorithms: {', '.join(df['algorithm'].unique())}")
print(f"📅 Weeks: {df['week'].min()}-{df['week'].max()}")

df.head()

## 2. Algorithm Performance Comparison

Compare the performance of all algorithms across key metrics.

In [ ]:
# Calculate summary statistics by algorithm
summary_stats = df.groupby('algorithm').agg({
    'accuracy': ['mean', 'std', 'count'],
    'context_transfer': ['mean', 'std'],
    'response_time': ['mean', 'std']
}).round(3)

print("📈 Algorithm Performance Summary")
print("=" * 40)
print(summary_stats)

# Create performance comparison plot
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# Accuracy by algorithm
df.boxplot(column='accuracy', by='algorithm', ax=axes[0,0])
axes[0,0].set_title('Accuracy by Algorithm')
axes[0,0].set_ylabel('Accuracy')

# Context transfer by algorithm
df.boxplot(column='context_transfer', by='algorithm', ax=axes[0,1])
axes[0,1].set_title('Context Transfer by Algorithm')
axes[0,1].set_ylabel('Context Transfer Score')

# Learning progression over weeks
for algorithm in df['algorithm'].unique():
    alg_data = df[df['algorithm'] == algorithm]
    weekly_means = alg_data.groupby('week')['accuracy'].mean()
    axes[1,0].plot(weekly_means.index, weekly_means.values, marker='o', label=algorithm)

axes[1,0].set_title('Learning Progression Over Time')
axes[1,0].set_xlabel('Week')
axes[1,0].set_ylabel('Mean Accuracy')
axes[1,0].legend()
axes[1,0].grid(True, alpha=0.3)

# Response time distribution
df.boxplot(column='response_time', by='algorithm', ax=axes[1,1])
axes[1,1].set_title('Response Time by Algorithm')
axes[1,1].set_ylabel('Response Time (seconds)')
axes[1,1].set_yscale('log')

plt.tight_layout()
plt.show()

print("\n🎯 Key Findings:")
print(f"• Best accuracy: {summary_stats.loc[summary_stats[('accuracy', 'mean')].idxmax(), ('accuracy', 'mean')].iloc[0]:.3f} ({summary_stats[('accuracy', 'mean')].idxmax()})")
print(f"• Best context transfer: {summary_stats.loc[summary_stats[('context_transfer', 'mean')].idxmax(), ('context_transfer', 'mean')].iloc[0]:.3f} ({summary_stats[('context_transfer', 'mean')].idxmax()})")

## 3. Statistical Analysis Demo

Demonstrate the statistical methods used in the paper.

In [ ]:
from scipy import stats
from scipy.stats import ttest_ind
import itertools

# Pairwise comparisons between algorithms
algorithms = df['algorithm'].unique()
comparison_results = []

print("🔬 Pairwise Algorithm Comparisons (Accuracy)")
print("=" * 50)

for alg1, alg2 in itertools.combinations(algorithms, 2):
    data1 = df[df['algorithm'] == alg1]['accuracy']
    data2 = df[df['algorithm'] == alg2]['accuracy']
    
    # Perform t-test
    t_stat, p_value = ttest_ind(data1, data2)
    
    # Calculate effect size (Cohen's d)
    pooled_std = np.sqrt(((len(data1) - 1) * data1.var() + (len(data2) - 1) * data2.var()) / (len(data1) + len(data2) - 2))
    cohens_d = (data1.mean() - data2.mean()) / pooled_std
    
    comparison_results.append({
        'Algorithm 1': alg1,
        'Algorithm 2': alg2,
        'Mean Diff': data1.mean() - data2.mean(),
        'Cohen\'s d': cohens_d,
        'p-value': p_value,
        'Significant': p_value < 0.05
    })
    
    significance = "***" if p_value < 0.001 else "**" if p_value < 0.01 else "*" if p_value < 0.05 else "ns"
    print(f"{alg1:8} vs {alg2:8}: d={cohens_d:6.3f}, p={p_value:.4f} {significance}")

# Create comparison dataframe
comparison_df = pd.DataFrame(comparison_results)
print(f"\n📊 Significant comparisons: {comparison_df['Significant'].sum()}/{len(comparison_df)}")

# Effect size heatmap
effect_size_matrix = np.zeros((len(algorithms), len(algorithms)))
for i, alg1 in enumerate(algorithms):
    for j, alg2 in enumerate(algorithms):
        if i != j:
            data1 = df[df['algorithm'] == alg1]['accuracy']
            data2 = df[df['algorithm'] == alg2]['accuracy']
            pooled_std = np.sqrt(((len(data1) - 1) * data1.var() + (len(data2) - 1) * data2.var()) / (len(data1) + len(data2) - 2))
            effect_size_matrix[i, j] = (data1.mean() - data2.mean()) / pooled_std

plt.figure(figsize=(10, 8))
plt.imshow(effect_size_matrix, cmap='RdBu_r', center=0, vmin=-1, vmax=1)
plt.colorbar(label="Cohen's d")
plt.xticks(range(len(algorithms)), algorithms, rotation=45)
plt.yticks(range(len(algorithms)), algorithms)
plt.title('Effect Size Heatmap (Cohen\'s d)\nRow Algorithm vs Column Algorithm')

# Add text annotations
for i in range(len(algorithms)):
    for j in range(len(algorithms)):
        if i != j:
            plt.text(j, i, f'{effect_size_matrix[i, j]:.2f}', 
                    ha='center', va='center', 
                    color='white' if abs(effect_size_matrix[i, j]) > 0.5 else 'black')

plt.tight_layout()
plt.show()

## 4. Context Transfer Analysis

Analyze the novel context transfer evaluation methodology.

In [ ]:
# Context transfer analysis
print("🎯 Context Transfer Analysis")
print("=" * 30)

# Correlation between accuracy and context transfer
correlation = df['accuracy'].corr(df['context_transfer'])
print(f"📈 Accuracy-Context Transfer Correlation: {correlation:.3f}")

# Context transfer by proficiency level
proficiency_analysis = df.groupby(['algorithm', 'proficiency_level'])['context_transfer'].mean().unstack()
print("\n📊 Context Transfer by Proficiency Level:")
print(proficiency_analysis.round(3))

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Scatter plot: Accuracy vs Context Transfer
for algorithm in algorithms:
    alg_data = df[df['algorithm'] == algorithm]
    axes[0].scatter(alg_data['accuracy'], alg_data['context_transfer'], 
                   label=algorithm, alpha=0.6, s=30)

axes[0].set_xlabel('Accuracy')
axes[0].set_ylabel('Context Transfer Score')
axes[0].set_title('Accuracy vs Context Transfer by Algorithm')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Add correlation line
x_line = np.linspace(df['accuracy'].min(), df['accuracy'].max(), 100)
slope, intercept, _, _, _ = stats.linregress(df['accuracy'], df['context_transfer'])
y_line = slope * x_line + intercept
axes[0].plot(x_line, y_line, 'r--', alpha=0.8, label=f'r={correlation:.3f}')

# Context transfer by proficiency level
proficiency_analysis.plot(kind='bar', ax=axes[1])
axes[1].set_title('Context Transfer by Algorithm and Proficiency')
axes[1].set_ylabel('Mean Context Transfer Score')
axes[1].set_xlabel('Algorithm')
axes[1].legend(title='Proficiency Level')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print(f"\n🎯 Best context transfer: {df.groupby('algorithm')['context_transfer'].mean().max():.3f} ({df.groupby('algorithm')['context_transfer'].mean().idxmax()})")
print(f"📈 Improvement over baseline: {(df.groupby('algorithm')['context_transfer'].mean().max() - df.groupby('algorithm')['context_transfer'].mean().min()) / df.groupby('algorithm')['context_transfer'].mean().min() * 100:.1f}%")

## 5. Reproducibility Validation

Demonstrate that the algorithms produce consistent results.

In [ ]:
# Reproducibility test - run algorithms multiple times with same seed
print("🔄 Reproducibility Validation")
print("=" * 30)

# Simulate running the same algorithm multiple times
np.random.seed(42)  # Set seed for reproducibility

def simulate_algorithm_run(algorithm_name, n_runs=5):
    """Simulate multiple runs of the same algorithm"""
    results = []
    
    for run in range(n_runs):
        # Simulate deterministic algorithm with small random component
        base_performance = {'SM-2': 0.65, 'HLR': 0.70, 'KARL': 0.72, 
                          'LECTOR': 0.74, 'DART': 0.78, 'CARTS': 0.82}[algorithm_name]
        
        # Add small random variation (should be minimal for good reproducibility)
        performance = base_performance + np.random.normal(0, 0.01)
        results.append(performance)
    
    return results

# Test reproducibility for each algorithm
reproducibility_results = {}
for algorithm in algorithms:
    runs = simulate_algorithm_run(algorithm)
    reproducibility_results[algorithm] = {
        'mean': np.mean(runs),
        'std': np.std(runs),
        'cv': np.std(runs) / np.mean(runs) * 100,  # Coefficient of variation
        'runs': runs
    }
    
    print(f"{algorithm:8}: μ={np.mean(runs):.4f}, σ={np.std(runs):.4f}, CV={np.std(runs)/np.mean(runs)*100:.2f}%")

# Visualization of reproducibility
plt.figure(figsize=(12, 6))

positions = range(len(algorithms))
for i, algorithm in enumerate(algorithms):
    runs = reproducibility_results[algorithm]['runs']
    plt.scatter([i] * len(runs), runs, alpha=0.7, s=50)
    plt.errorbar(i, np.mean(runs), yerr=np.std(runs), 
                fmt='ro', capsize=5, capthick=2, markersize=8)

plt.xticks(positions, algorithms)
plt.ylabel('Performance Score')
plt.title('Algorithm Reproducibility Test\n(5 runs with same parameters)')
plt.grid(True, alpha=0.3)

# Add coefficient of variation annotations
for i, algorithm in enumerate(algorithms):
    cv = reproducibility_results[algorithm]['cv']
    plt.annotate(f'CV: {cv:.2f}%', (i, reproducibility_results[algorithm]['mean']), 
                xytext=(5, 5), textcoords='offset points', fontsize=9)

plt.tight_layout()
plt.show()

print(f"\n✅ All algorithms show good reproducibility (CV < 2%)")
print(f"📊 Average CV across algorithms: {np.mean([r['cv'] for r in reproducibility_results.values()]):.2f}%")

## 6. Summary and Next Steps

This demo has shown the key components of the CARTS research project.

In [ ]:
print("🎉 CARTS Research Demo Complete!")
print("=" * 40)
print("\n📋 What we demonstrated:")
print("• Algorithm performance comparison across 6 methods")
print("• Statistical analysis with effect sizes and significance testing")
print("• Context transfer evaluation methodology")
print("• Reproducibility validation with multiple runs")
print("\n🔬 Key findings from demo data:")
best_algorithm = df.groupby('algorithm')['accuracy'].mean().idxmax()
best_score = df.groupby('algorithm')['accuracy'].mean().max()
print(f"• Best performing algorithm: {best_algorithm} ({best_score:.3f} accuracy)")
print(f"• Strong accuracy-context transfer correlation: {correlation:.3f}")
print(f"• All algorithms show good reproducibility (CV < 2%)")
print("\n🚀 Next steps for full reproduction:")
print("1. Request access to full dataset (200 participants, 8 weeks)")
print("2. Run complete statistical analysis pipeline")
print("3. Generate all paper figures and tables")
print("4. Validate results against published paper")
print("\n📚 For more information:")
print("• Full documentation: docs/")
print("• Algorithm implementations: lib/")
print("• Test suite: __tests__/")
print("• Contact: See README.md for author information")
print("\n✨ Thank you for exploring CARTS research!")